# m3 — Structural Features

Per-sample: ColabFold structure prediction → pLDDT/pTM extraction → physicochemical properties →
mCSM-PDB2 ΔΔG → MAESTRO ΔΔG → per-sample summary CSV.
Then assembles the flat **feature table** used by Module 4.

| Step | Tool | GPU |
|------|------|-----|
| 5b | ColabFold / AlphaFold2 | **Yes** |
| 6a | pLDDT + pTM extraction | No |
| 6b | KD, Grantham, BLOSUM62, charge, polarity, size | No |
| 6c | mCSM-PDB2 ΔΔG via API | No |
| 6d | MAESTRO ΔΔG via API | No |
| 6e | Per-sample summary CSV + pLDDT plot | No |
| M3 | Feature table assembly | No |

**Prerequisite**: `colabfold_ready.flag` must exist on Drive (run `m0_setup_and_discovery.ipynb`).

In [ ]:
# parameters
SAMPLE_CSV         = ""
SRR_LIST           = []
SRR_ACCESSION      = ""
DRIVE_OUTPUT       = "mmpR5_pipeline/output"
DRIVE_REF          = "mmpR5_pipeline/input"
WT_PDB_ID          = "4NB8"
MCSM_CHAIN         = "A"
RUN_COLABFOLD      = True
RUN_MCSM           = True
RUN_MAESTRO        = True

In [ ]:
# CPU only — no GPU needed
# ── Load pipeline config (Drive JSON fallback) ──────────────────────────────
import json, shutil, subprocess, warnings, time, datetime, concurrent.futures
from pathlib import Path
from Bio import SeqIO, Entrez
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
import pandas as pd
import numpy as np
import requests
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

_CFG_PATH = Path("/content/drive/MyDrive/ColabNotebooks/mmpR5_pipeline/pipeline_config.json")

def _load_config():
    if _CFG_PATH.exists():
        with open(_CFG_PATH) as _f:
            return json.load(_f)
    return {}

_cfg = _load_config()

def _p(key, default=None):
    """Resolve parameter: papermill-injected variable takes precedence over config JSON."""
    try:
        v = eval(key)                # injected by papermill
        return v if v is not None else _cfg.get(key, default)
    except Exception:
        return _cfg.get(key, default)

In [ ]:
# CPU only — no GPU needed
from google.colab import drive
drive.mount("/content/drive")
DRIVE_BASE  = Path("/content/drive/MyDrive/ColabNotebooks")
OUTPUT_ROOT = DRIVE_BASE / _p("DRIVE_OUTPUT", "mmpR5_pipeline/output")
REF_DIR     = DRIVE_BASE / _p("DRIVE_REF",    "mmpR5_pipeline/input")
MODULES_DIR = DRIVE_BASE / "mmpR5_pipeline" / "modules"
print(f"Drive mounted. Output root: {OUTPUT_ROOT}")

In [ ]:
# CPU only — no GPU needed
# ── Check ColabFold sentinel ──────────────────────────────────────────────────
_sentinel = DRIVE_BASE / "mmpR5_pipeline" / "colabfold_ready.flag"
if not _sentinel.exists():
    raise RuntimeError(
        "ColabFold is not installed.\n"
        "Run m0_setup_and_discovery.ipynb manually and restart the runtime\n"
        "before running m3 or the controller."
    )
print(f"ColabFold sentinel: {_sentinel.read_text().strip()}")

# ── Resolve parameters ────────────────────────────────────────────────────────
_WT_PDB_ID   = _p("WT_PDB_ID",   "4NB8")
_MCSM_CHAIN  = _p("MCSM_CHAIN",  "A")
_RUN_CF      = bool(_p("RUN_COLABFOLD", True))
_RUN_MCSM    = bool(_p("RUN_MCSM",    True))
_RUN_MAESTRO = bool(_p("RUN_MAESTRO", True))

# Resolve sample manifest
_manifest_csv = OUTPUT_ROOT / "sample_manifest.csv"
if _manifest_csv.exists():
    _mdf = pd.read_csv(str(_manifest_csv))
    _sample_manifest = [{**r, "label": str(r.get("sample_label", r.get("srr","")))} for r in _mdf.to_dict("records")]
else:
    _sample_csv = _p("SAMPLE_CSV",""); _srr_list = _p("SRR_LIST",[]); _srr_single = _p("SRR_ACCESSION","")
    _sample_manifest = []
    if _sample_csv and Path(_sample_csv).exists():
        for _,_r in pd.read_csv(_sample_csv).iterrows():
            _sample_manifest.append({"srr":str(_r["srr"]),"label":str(_r.get("sample_label",_r["srr"])),"phenotype":str(_r.get("phenotype","U"))})
    elif _srr_list:
        _sample_manifest = [{"srr":s,"label":s,"phenotype":"U"} for s in _srr_list]
    elif _srr_single:
        _sample_manifest = [{"srr":_srr_single,"label":_srr_single,"phenotype":"U"}]
    else:
        raise ValueError("No samples.")

print(f"Samples: {len(_sample_manifest)}")

In [ ]:
# CPU only — no GPU needed
# ── Physicochemical property tables ──────────────────────────────────────────
_KD = {"A":1.8,"R":-4.5,"N":-3.5,"D":-3.5,"C":2.5,"Q":-3.5,"E":-3.5,"G":-0.4,
       "H":-3.2,"I":4.5,"L":3.8,"K":-3.9,"M":1.9,"F":2.8,"P":-1.6,"S":-0.8,
       "T":-0.7,"W":-0.9,"Y":-1.3,"V":4.2,"*":0,"X":0}
_GRANTHAM = {}
for _a,_b,_s in [
    ("A","D",126),("A","E",107),("A","F",113),("A","G",60),("A","H",86),("A","I",94),
    ("A","K",106),("A","L",96),("A","M",84),("A","N",111),("A","P",27),("A","Q",91),
    ("A","R",112),("A","S",99),("A","T",58),("A","V",64),("A","W",148),("A","Y",112),
    ("A","C",195),("D","E",45),("D","F",177),("D","G",94),("D","H",81),("D","I",168),
    ("D","K",101),("D","L",172),("D","M",160),("D","N",23),("D","P",108),("D","Q",61),
    ("D","R",96),("D","S",65),("D","T",85),("D","V",152),("D","W",181),("D","Y",160),
    ("D","C",154),("E","F",140),("E","G",98),("E","H",40),("E","I",134),("E","K",56),
    ("E","L",138),("E","M",126),("E","N",42),("E","P",93),("E","Q",29),("E","R",54),
    ("E","S",80),("E","T",65),("E","V",121),("E","W",152),("E","Y",122),("E","C",170),
    ("F","G",153),("F","H",100),("F","I",21),("F","K",102),("F","L",22),("F","M",28),
    ("F","N",158),("F","P",114),("F","Q",116),("F","R",97),("F","S",155),("F","T",103),
    ("F","V",50),("F","W",40),("F","Y",22),("F","C",205),("G","H",98),("G","I",135),
    ("G","K",127),("G","L",138),("G","M",127),("G","N",80),("G","P",42),("G","Q",87),
    ("G","R",125),("G","S",56),("G","T",59),("G","V",109),("G","W",184),("G","Y",147),
    ("G","C",159),("H","I",94),("H","K",32),("H","L",99),("H","M",87),("H","N",68),
    ("H","P",77),("H","Q",24),("H","R",29),("H","S",89),("H","T",47),("H","V",84),
    ("H","W",115),("H","Y",83),("H","C",174),("I","K",102),("I","L",5),("I","M",10),
    ("I","N",149),("I","P",95),("I","Q",109),("I","R",97),("I","S",142),("I","T",89),
    ("I","V",29),("I","W",61),("I","Y",33),("I","C",198),("K","L",102),("K","M",95),
    ("K","N",94),("K","P",103),("K","Q",53),("K","R",26),("K","S",121),("K","T",78),
    ("K","V",97),("K","W",110),("K","Y",85),("K","C",202),("L","M",15),("L","N",153),
    ("L","P",98),("L","Q",113),("L","R",102),("L","S",145),("L","T",92),("L","V",32),
    ("L","W",61),("L","Y",36),("L","C",198),("M","N",142),("M","P",87),("M","Q",101),
    ("M","R",91),("M","S",135),("M","T",81),("M","V",21),("M","W",67),("M","Y",36),
    ("M","C",196),("N","P",91),("N","Q",46),("N","R",86),("N","S",46),("N","T",65),
    ("N","V",133),("N","W",174),("N","Y",143),("N","C",139),("P","Q",76),("P","R",103),
    ("P","S",74),("P","T",38),("P","V",68),("P","W",147),("P","Y",110),("P","C",169),
    ("Q","R",43),("Q","S",68),("Q","T",42),("Q","V",96),("Q","W",130),("Q","Y",99),
    ("Q","C",154),("R","S",110),("R","T",71),("R","V",96),("R","W",101),("R","Y",77),
    ("R","C",180),("S","T",58),("S","V",124),("S","W",177),("S","Y",144),("S","C",112),
    ("T","V",69),("T","W",128),("T","Y",92),("T","C",149),("V","W",88),("V","Y",55),
    ("V","C",192),("W","Y",37),("W","C",215),("Y","C",194)]:
    _GRANTHAM[frozenset([_a,_b])] = _s
_BLOSUM62 = {
    "A":{"A":4,"R":-1,"N":-2,"D":-2,"C":0,"Q":-1,"E":-1,"G":0,"H":-2,"I":-1,"L":-1,"K":-1,"M":-1,"F":-2,"P":-1,"S":1,"T":0,"W":-3,"Y":-2,"V":0},
    "R":{"A":-1,"R":5,"N":0,"D":-2,"C":-3,"Q":1,"E":0,"G":-2,"H":0,"I":-3,"L":-2,"K":2,"M":-1,"F":-3,"P":-2,"S":-1,"T":-1,"W":-3,"Y":-2,"V":-3},
    "N":{"A":-2,"R":0,"N":6,"D":1,"C":-3,"Q":0,"E":0,"G":0,"H":1,"I":-3,"L":-3,"K":0,"M":-2,"F":-3,"P":-2,"S":1,"T":0,"W":-4,"Y":-2,"V":-3},
    "D":{"A":-2,"R":-2,"N":1,"D":6,"C":-3,"Q":0,"E":2,"G":-1,"H":-1,"I":-3,"L":-4,"K":-1,"M":-3,"F":-3,"P":-1,"S":0,"T":-1,"W":-4,"Y":-3,"V":-3},
    "C":{"A":0,"R":-3,"N":-3,"D":-3,"C":9,"Q":-3,"E":-4,"G":-3,"H":-3,"I":-1,"L":-1,"K":-3,"M":-1,"F":-2,"P":-3,"S":-1,"T":-1,"W":-2,"Y":-2,"V":-1},
    "Q":{"A":-1,"R":1,"N":0,"D":0,"C":-3,"Q":5,"E":2,"G":-2,"H":0,"I":-3,"L":-2,"K":1,"M":0,"F":-3,"P":-1,"S":0,"T":-1,"W":-2,"Y":-1,"V":-2},
    "E":{"A":-1,"R":0,"N":0,"D":2,"C":-4,"Q":2,"E":5,"G":-2,"H":0,"I":-3,"L":-3,"K":1,"M":-2,"F":-3,"P":-1,"S":0,"T":-1,"W":-3,"Y":-2,"V":-2},
    "G":{"A":0,"R":-2,"N":0,"D":-1,"C":-3,"Q":-2,"E":-2,"G":6,"H":-2,"I":-4,"L":-4,"K":-2,"M":-3,"F":-3,"P":-2,"S":0,"T":-2,"W":-2,"Y":-3,"V":-3},
    "H":{"A":-2,"R":0,"N":1,"D":-1,"C":-3,"Q":0,"E":0,"G":-2,"H":8,"I":-3,"L":-3,"K":-1,"M":-2,"F":-1,"P":-2,"S":-1,"T":-2,"W":-2,"Y":2,"V":-3},
    "I":{"A":-1,"R":-3,"N":-3,"D":-3,"C":-1,"Q":-3,"E":-3,"G":-4,"H":-3,"I":4,"L":2,"K":-3,"M":1,"F":0,"P":-3,"S":-2,"T":-1,"W":-3,"Y":-1,"V":3},
    "L":{"A":-1,"R":-2,"N":-3,"D":-4,"C":-1,"Q":-2,"E":-3,"G":-4,"H":-3,"I":2,"L":4,"K":-2,"M":2,"F":0,"P":-3,"S":-2,"T":-1,"W":-2,"Y":-1,"V":1},
    "K":{"A":-1,"R":2,"N":0,"D":-1,"C":-3,"Q":1,"E":1,"G":-2,"H":-1,"I":-3,"L":-2,"K":5,"M":-1,"F":-3,"P":-1,"S":0,"T":-1,"W":-3,"Y":-2,"V":-2},
    "M":{"A":-1,"R":-1,"N":-2,"D":-3,"C":-1,"Q":0,"E":-2,"G":-3,"H":-2,"I":1,"L":2,"K":-1,"M":5,"F":0,"P":-2,"S":-1,"T":-1,"W":-1,"Y":-1,"V":1},
    "F":{"A":-2,"R":-3,"N":-3,"D":-3,"C":-2,"Q":-3,"E":-3,"G":-3,"H":-1,"I":0,"L":0,"K":-3,"M":0,"F":6,"P":-4,"S":-2,"T":-2,"W":1,"Y":3,"V":-1},
    "P":{"A":-1,"R":-2,"N":-2,"D":-1,"C":-3,"Q":-1,"E":-1,"G":-2,"H":-2,"I":-3,"L":-3,"K":-1,"M":-2,"F":-4,"P":7,"S":-1,"T":-1,"W":-4,"Y":-3,"V":-2},
    "S":{"A":1,"R":-1,"N":1,"D":0,"C":-1,"Q":0,"E":0,"G":0,"H":-1,"I":-2,"L":-2,"K":0,"M":-1,"F":-2,"P":-1,"S":4,"T":1,"W":-3,"Y":-2,"V":-2},
    "T":{"A":0,"R":-1,"N":0,"D":-1,"C":-1,"Q":-1,"E":-1,"G":-2,"H":-2,"I":-1,"L":-1,"K":-1,"M":-1,"F":-2,"P":-1,"S":1,"T":5,"W":-2,"Y":-2,"V":0},
    "W":{"A":-3,"R":-3,"N":-4,"D":-4,"C":-2,"Q":-2,"E":-3,"G":-2,"H":-2,"I":-3,"L":-2,"K":-3,"M":-1,"F":1,"P":-4,"S":-3,"T":-2,"W":11,"Y":2,"V":-3},
    "Y":{"A":-2,"R":-2,"N":-2,"D":-3,"C":-2,"Q":-1,"E":-2,"G":-3,"H":2,"I":-1,"L":-1,"K":-2,"M":-1,"F":3,"P":-3,"S":-2,"T":-2,"W":2,"Y":7,"V":-1},
    "V":{"A":0,"R":-3,"N":-3,"D":-3,"C":-1,"Q":-2,"E":-2,"G":-3,"H":-3,"I":3,"L":1,"K":-2,"M":1,"F":-1,"P":-2,"S":-2,"T":0,"W":-3,"Y":-1,"V":4}}
_AA_CHARGE = {"R":1,"K":1,"H":0.1,"D":-1,"E":-1,"A":0,"C":0,"F":0,"G":0,"I":0,
              "L":0,"M":0,"N":0,"P":0,"Q":0,"S":0,"T":0,"V":0,"W":0,"Y":0,"*":0,"X":0}
_AA_POLAR  = {"R","K","H","D","E","N","Q","S","T","Y"}
_AA_SIZE   = {"G":"tiny","A":"small","S":"small","T":"small","C":"small","V":"medium",
              "P":"medium","I":"medium","L":"medium","M":"medium","D":"medium","N":"medium",
              "E":"large","Q":"large","K":"large","R":"large","H":"large","F":"large",
              "Y":"large","W":"large","*":"none","X":"none"}
_MCSM_URL   = "http://biosig.unimelb.edu.au/mcsm_pdb2/api/prediction"
_MAESTRO_URL = "https://biwww.che.sbg.ac.at/MAESTRO/web/api/prediction"
print("Property tables loaded.")

In [ ]:
# ── Scoring functions ──────────────────────────────────────────────────────────
def _get_wt_pdb(wdir):
    wt_pdb = wdir / f"{_WT_PDB_ID}.pdb"
    if not wt_pdb.exists():
        r = requests.get(f"https://files.rcsb.org/download/{_WT_PDB_ID}.pdb", timeout=30)
        if r.status_code == 200:
            wt_pdb.write_bytes(r.content)
    return wt_pdb if wt_pdb.exists() else None

def _phys(aa_df):
    rows = []
    for _, row in aa_df.iterrows():
        wt = row.get("wt_aa","X"); mut = row.get("mut_aa","X")
        if row.get("effect") in ("indel",None) or wt=="?" or mut=="?": continue
        rows.append({"aa_change":row.get("hgvs_p","?"),"effect":row.get("effect","?"),
            "delta_kd":round(_KD.get(mut,0)-_KD.get(wt,0),2),
            "grantham":_GRANTHAM.get(frozenset([wt,mut]),0 if wt==mut else "N/A"),
            "blosum62":_BLOSUM62.get(wt,{}).get(mut,"N/A"),
            "delta_charge":round(_AA_CHARGE.get(mut,0)-_AA_CHARGE.get(wt,0),2),
            "polarity_change":(wt in _AA_POLAR)!=(mut in _AA_POLAR),
            "wt_size":_AA_SIZE.get(wt,"?"),"mut_size":_AA_SIZE.get(mut,"?"),
            "size_change":_AA_SIZE.get(wt,"?")!=_AA_SIZE.get(mut,"?")})
    return pd.DataFrame(rows)

def _mcsm(aa_df, wdir):
    res = {}; wt_pdb = _get_wt_pdb(wdir)
    if wt_pdb is None: return res
    for _, row in aa_df[aa_df["effect"].isin(["missense","nonsense"])].iterrows():
        ms = f"{_MCSM_CHAIN}_{row['wt_aa']}{row['aa_pos']}{row['mut_aa']}"
        try:
            with open(wt_pdb,"rb") as fh:
                resp = requests.post(_MCSM_URL,
                    files={"pdb_file":(wt_pdb.name,fh,"chemical/x-pdb")},
                    data={"mutation":ms}, timeout=120)
            if resp.status_code == 200:
                d = resp.json()
                res[ms] = {"ddg":d.get("prediction") or d.get("ddg") or d.get("ddG")}
            else:
                res[ms] = {"ddg":None,"error":f"HTTP {resp.status_code}"}
        except Exception as exc:
            res[ms] = {"ddg":None,"error":str(exc)}
        time.sleep(2)
    if not any(v.get("ddg") for v in res.values()):
        print("  mCSM fallback: http://biosig.unimelb.edu.au/mcsm_pdb2/")
    return res

def _maestro(aa_df, wdir):
    res = {}; wt_pdb = _get_wt_pdb(wdir)
    if wt_pdb is None: return res
    for _, row in aa_df[aa_df["effect"].isin(["missense","nonsense"])].iterrows():
        ms = f"{row['wt_aa']}{row['aa_pos']}{row['mut_aa']}"
        try:
            with open(wt_pdb,"rb") as fh:
                resp = requests.post(_MAESTRO_URL,
                    files={"pdb":(wt_pdb.name,fh,"chemical/x-pdb")},
                    data={"mutation":f"{_MCSM_CHAIN}:{ms}","ensemble":"1"}, timeout=120)
            if resp.status_code == 200:
                d = resp.json() if resp.headers.get("content-type","").startswith("application/json") else {}
                res[ms] = {"ddg":d.get("ddg") or d.get("prediction") or d.get("dG")}
            else:
                res[ms] = {"ddg":None,"error":f"HTTP {resp.status_code}"}
        except Exception as exc:
            res[ms] = {"ddg":None,"error":str(exc)}
        time.sleep(2)
    if not any(v.get("ddg") for v in res.values()):
        print("  MAESTRO fallback: https://biwww.che.sbg.ac.at/MAESTRO/")
    return res

In [ ]:
# ── Batch loop: ColabFold + scoring ──────────────────────────────────────────
if _RUN_CF:
    try:
        import alphafold, jax
        _gpu = [d for d in jax.devices() if "cuda" in str(d).lower() or "gpu" in str(d).lower()]
        if not _gpu:
            raise RuntimeError("No GPU detected. Change runtime type to T4 GPU.")
        print(f"JAX {jax.__version__} | GPU: {_gpu[0]}")
    except ModuleNotFoundError:
        raise RuntimeError("alphafold not found. Run m0_setup_and_discovery.ipynb and restart runtime.")
else:
    print("RUN_COLABFOLD=False — skipping structure prediction")

import glob as _g
_m3_status = {}

for _sm in _sample_manifest:
    _srr, _label = _sm["srr"], _sm["label"]
    _wdir = Path(f"/content/{_label}"); _odir = OUTPUT_ROOT / _label
    print(f"{'─'*65}")
    _sumcsv = _odir/"06_scores"/f"{_label}_full_summary.csv"
    if _sumcsv.exists():
        print(f"[SKIP] {_label} — full_summary.csv exists"); _m3_status[_label]="DONE"; continue

    # Restore aa_df from Drive
    _aa_tsv = _wdir/f"{_label}_aa_annotation.tsv"
    if not _aa_tsv.exists():
        _src = _odir/"04_sequences"/f"{_label}_aa_annotation.tsv"
        if _src.exists(): _wdir.mkdir(parents=True,exist_ok=True); shutil.copy2(_src,_aa_tsv)
    if not _aa_tsv.exists():
        print(f"[SKIP] {_label} — AA annotation missing; run m2 first"); continue
    _aa_df = pd.read_csv(str(_aa_tsv),sep="\t")
    _aa_fa  = _wdir/f"{_label}_mmpR5_protein.fasta"
    if not _aa_fa.exists():
        _src2 = _odir/"04_sequences"/f"{_label}_mmpR5_protein.fasta"
        if _src2.exists(): shutil.copy2(_src2,_aa_fa)
    if not _aa_fa.exists():
        print(f"[SKIP] {_label} — protein FASTA missing"); continue

    print(f"[RUN] {_label}")
    _wdir.mkdir(parents=True,exist_ok=True)
    for _s in ["05_structure","06_scores"]:
        (_odir/_s).mkdir(parents=True,exist_ok=True)
    _aa_seq = str(SeqIO.read(str(_aa_fa),"fasta").seq)
    _mut_label = "wild-type"
    if not _aa_df.empty:
        _mut_label = ", ".join(row["hgvs_p"] if row["effect"]!="indel" else row["hgvs_c"]
                               for _,row in _aa_df.iterrows())
    try:
        # ColabFold
        _cf_out = _wdir/"colabfold_output"; _cf_out.mkdir(exist_ok=True)
        _best_pdb = None
        if _RUN_CF:
            if not list(_cf_out.glob("*.pdb")):
                _cf_in = _wdir/f"{_label}_mmpR5_for_colabfold.fasta"
                with open(str(_cf_in),"w") as fh: fh.write(f">{_label}_mmpR5\n{_aa_seq}\n")
                res = subprocess.run(["colabfold_batch",str(_cf_in),str(_cf_out)+"/",
                    "--num-recycle","3","--model-type","alphafold2_ptm","--use-gpu-relax"],
                    capture_output=True, text=True)
                if res.returncode != 0: print(f"  ColabFold stderr: {res.stderr[-300:]}")
            else:
                print(f"  [{_label}] ColabFold output cached")
            for f in _cf_out.glob("*"):
                shutil.copy2(f,_odir/"05_structure"/f.name)
            _bp = sorted(_cf_out.glob("*rank_001*.pdb"))
            _best_pdb = _bp[0] if _bp else None

        # pLDDT
        _cf_scores = []
        for _jp in sorted(_g.glob(str(_cf_out/"*_scores_rank_001_*.json"))):
            with open(_jp) as fh: _jd = json.load(fh)
            _pl = _jd.get("plddt",[])
            _cf_scores.append({"avg_plddt":round(sum(_pl)/len(_pl),2) if _pl else None,
                "ptm":round(_jd.get("ptm"),4) if _jd.get("ptm") else None, "plddt_per_residue":_pl})
        _cf_best = _cf_scores[0] if _cf_scores else {}
        if _cf_best: print(f"  [{_label}] pLDDT={_cf_best.get('avg_plddt')}  pTM={_cf_best.get('ptm')}")

        # Physicochemical
        _phys_df = _phys(_aa_df)
        _phys_csv = _wdir/f"{_label}_physicochemical.csv"; _phys_df.to_csv(str(_phys_csv),index=False)
        shutil.copy2(_phys_csv,_odir/"06_scores"/_phys_csv.name)

        # mCSM
        _mcsm_r = _mcsm(_aa_df,_wdir) if _RUN_MCSM else {}

        # MAESTRO
        _maestro_r = _maestro(_aa_df,_wdir) if _RUN_MAESTRO else {}

        # Per-sample summary
        _rows = []
        for _,row in _aa_df.iterrows():
            _ms  = f"{_MCSM_CHAIN}_{row.get('wt_aa','?')}{row.get('aa_pos','?')}{row.get('mut_aa','?')}"
            _ms2 = f"{row.get('wt_aa','?')}{row.get('aa_pos','?')}{row.get('mut_aa','?')}"
            _ph  = _phys_df[_phys_df["aa_change"]==row.get("hgvs_p","")].to_dict("records")
            _ph  = _ph[0] if _ph else {}
            _rows.append({"Sample":_label,"Mutation (HGVS_p)":row.get("hgvs_p","wild-type"),
                "Mutation (HGVS_c)":row.get("hgvs_c",""),"Effect":row.get("effect",""),
                "AA_pos":row.get("aa_pos",""),"WT_aa":row.get("wt_aa",""),"Mut_aa":row.get("mut_aa",""),
                "avg_pLDDT":_cf_best.get("avg_plddt",""),"pTM":_cf_best.get("ptm",""),
                "delta_KD_hydrophobicity":_ph.get("delta_kd",""),"Grantham_score":_ph.get("grantham",""),
                "BLOSUM62":_ph.get("blosum62",""),"delta_charge":_ph.get("delta_charge",""),
                "polarity_change":_ph.get("polarity_change",""),"size_change":_ph.get("size_change",""),
                "mCSM_ddG_stability":_mcsm_r.get(_ms,{}).get("ddg","not run"),
                "MAESTRO_ddG":_maestro_r.get(_ms2,{}).get("ddg","not run")})
        if not _rows:
            _rows = [{"Sample":_label,"Mutation (HGVS_p)":"wild-type",
                      "avg_pLDDT":_cf_best.get("avg_plddt",""),"pTM":_cf_best.get("ptm","")}]
        _sum_df = pd.DataFrame(_rows)
        _sum_csv = _wdir/f"{_label}_full_summary.csv"; _sum_df.to_csv(str(_sum_csv),index=False)

        # pLDDT plot
        if _cf_scores and _cf_scores[0].get("plddt_per_residue"):
            _pv = _cf_scores[0]["plddt_per_residue"]
            fig,ax = plt.subplots(figsize=(12,3))
            ax.bar(range(1,len(_pv)+1),_pv,
                   color=["#003f88" if v>=90 else "#00b4d8" if v>=70 else "#fca311" if v>=50 else "#e63946" for v in _pv],width=1.0)
            for t,c,l in [(90,"#003f88",">90"),(70,"#00b4d8","70–90"),(50,"#fca311","50–70")]:
                ax.axhline(t,color=c,ls="--",lw=0.8,label=l)
            ax.set_ylim(0,100); ax.set_xlabel("Residue"); ax.set_ylabel("pLDDT")
            ax.set_title(f"{_label} | {_mut_label}"); ax.legend(fontsize=8,loc="lower right")
            plt.tight_layout()
            _pp = _wdir/f"{_label}_plddt_plot.png"; plt.savefig(str(_pp),dpi=150); plt.close()

        # Copy to Drive
        for _f in [_sum_csv, _wdir/f"{_label}_plddt_plot.png"]:
            if Path(_f).exists(): shutil.copy2(_f,_odir/"06_scores"/Path(_f).name)
        if _best_pdb and Path(str(_best_pdb)).exists():
            shutil.copy2(str(_best_pdb),_odir/"06_scores"/Path(str(_best_pdb)).name)
        _m3_status[_label] = "DONE"; print(f"[DONE] {_label}")
    except Exception as _exc:
        import traceback; traceback.print_exc()
        _m3_status[_label] = f"FAILED: {_exc}"

# ── Feature table assembly ────────────────────────────────────────────────────
print("\n" + "="*70)
print("  FEATURE TABLE ASSEMBLY")
_ft_rows = []
for _sm in _sample_manifest:
    _label = _sm["label"]; _phen = _sm.get("phenotype","U")
    _sc = OUTPUT_ROOT/_sm["label"]/"06_scores"/f"{_sm['label']}_full_summary.csv"
    if not _sc.exists(): print(f"  [MISSING] {_label}"); continue
    _df = pd.read_csv(str(_sc)); _df.insert(0,"SRR",_sm["srr"]); _df.insert(1,"sample_label",_label); _df.insert(2,"phenotype",_phen)
    _ft_rows.append(_df)
if _ft_rows:
    _ft = pd.concat(_ft_rows,ignore_index=True)
    for _c in ["mCSM_ddG_stability","MAESTRO_ddG"]: _ft[_c] = pd.to_numeric(_ft[_c],errors="coerce")
    _ft["has_structural_data"] = _ft["mCSM_ddG_stability"].notna() | _ft["MAESTRO_ddG"].notna()
    def _etype(r):
        h=str(r.get("Mutation (HGVS_p)","")).strip(); e=str(r.get("Effect","")).strip()
        if h in ("wild-type","") and e in ("","nan","None"): return "WT"
        if e not in ("","nan","None"): return e
        return "frameshift" if ("indel" in h.lower() or "fs" in h.lower()) else "unknown"
    _ft["effect_type"] = _ft.apply(_etype,axis=1)
    _ft_csv = OUTPUT_ROOT/"feature_table_all_samples.csv"; _ft.to_csv(str(_ft_csv),index=False)
    print(f"  Feature table: {len(_ft)} rows x {len(_ft.columns)} cols → {_ft_csv.name}")
    display(_ft.head())